# Episode 7 — ML Optimizers

**Student workbook** · code along with the video.

Implement **SGD → Momentum → RMSProp → AdamW → Muon** as pure-JAX `NamedTuple`s
(`init` + `__call__`). Model, data, and the train loop live in [`helpers.py`](./helpers.py);
**you write the optimizers here.**

| | |
|---|---|
| **Chapter** | 2.2 · Part II — GPT-2 & single-GPU training |
| **Prereq** | [Episode 6](../ep06/solution.ipynb) |
| **Next** | Episode 8 — Memory & mixed precision |

**Visual intuition:**

1. [Why Momentum Really Works](https://distill.pub/2017/momentum/) — Distill on momentum, step-size, curvature
2. [Gradient Optimizer Comparison](https://www.corefranciscopark.com/blog/gradient-optimizer-comparison) — race optimizers on classic loss landscapes


## Setup

[`helpers.py`](./helpers.py) provides Tiny Shakespeare loading, the GPT model, `train(...)`,
and Muon utilities (`map_leaves`, `newton_schulz5`).


In [ ]:
# your code here


## Data + model

Tiny Shakespeare, ~16M GPT (4×256×8).


In [ ]:
# your code here


## Optimizer contract

Every optimizer is a `NamedTuple` of hyperparameters with:

```python
opt_state = optimizer.init(params)                    # allocate state (or None)
params, opt_state = optimizer(params, grads, opt_state, step)  # step is 1-indexed
```

`train(..., optimizer=opt)` only needs `.name`, `.learning_rate`, `.init`, and `__call__`.


## 1. SGD

$$
\theta \leftarrow \theta - \eta\, g
$$

| Pros | Cons |
|------|------|
| Dead simple; no state; easy to reason about | Slow on ill-conditioned / sparse-grad problems |
| Strong baseline when LR is tuned (often with schedule) | Sensitive to learning rate; zigzags in narrow valleys |
| Often best *final* convergence given enough time | No per-coordinate scaling |

See also the Distill piece on why plain GD struggles with pathological curvature.


### Implement `SGD`



In [ ]:
# your code here


## 2. Momentum (heavy-ball)

$$
\begin{aligned}
v &\leftarrow \beta\, v + g \\
\theta &\leftarrow \theta - \eta\, v
\end{aligned}
$$

| Pros | Cons |
|------|------|
| Accelerates along consistent directions; damps oscillation | Extra state (velocity tree = params size) |
| Larger usable LR range than plain SGD | Can overshoot sharp minima if β / η too large |
| Great intuition in [Distill — Momentum](https://distill.pub/2017/momentum/) | Still no per-coordinate adaptation |

Try the Distill sliders: β near 0.99 on a skinny quadratic.


### Implement `Momentum`



In [ ]:
# your code here


## 3. RMSProp

$$
\begin{aligned}
v &\leftarrow \rho\, v + (1-\rho)\, g^{2} \\
\theta &\leftarrow \theta - \eta\, \frac{g}{\sqrt{v}+\varepsilon}
\end{aligned}
$$

| Pros | Cons |
|------|------|
| Per-coordinate step sizes — good for sparse / varying scales | Still first-order; can be noisy |
| Stabilizes training when gradient magnitudes differ a lot | Two hyperparameters (ρ, η); ε matters numerically |
| Cheap: one second-moment tree | No momentum on the gradient itself (unlike Adam) |


### Implement `RMSProp`



In [ ]:
# your code here


## 4. AdamW

**Adam core** (bias-corrected moments), then **decoupled weight decay**
$\theta \leftarrow \theta\,(1-\eta\lambda)$ *before* the Adam step (not L2 folded into $g$):

$$
\begin{aligned}
m &\leftarrow \beta_1 m + (1-\beta_1)\, g \\
v &\leftarrow \beta_2 v + (1-\beta_2)\, g^{2} \\
\hat{m} &= \frac{m}{1-\beta_1^{t}},\quad
\hat{v} = \frac{v}{1-\beta_2^{t}} \\
\theta &\leftarrow \theta\,(1-\eta\lambda)
         - \eta\, \frac{\hat{m}}{\sqrt{\hat{v}}+\varepsilon}
\end{aligned}
$$

| Pros | Cons |
|------|------|
| Default for transformers; robust across LRs | 2× moment memory vs SGD |
| Combines momentum + RMSProp-style scaling | Can under-regularize if you use L2-in-loss instead of decoupled decay |
| Bias correction helps early training | Still diagonal — ignores matrix structure of weight grads |
| Decoupled decay usually beats “Adam + L2” | Easy to overfit tiny corpora without dropout / schedules |

**Pretraining tip:** lock in peak LR (and the warmup → decay schedule) on short ablations *before* the long run. Once cosine/linear decay is underway, bumping LR mid-flight usually destabilizes Adam’s moment scales — treat a bad peak LR as a restart, not a mid-run tweak.


### Implement `AdamState` + `AdamW`

`step` is **1-indexed** (matches `train`).



In [ ]:
# your code here


## 5. Muon

For **2D** weights: momentum, then Newton–Schulz orthogonalization
(`newton_schulz5` ≈ nearest semi-orthogonal matrix $\approx UV^{\top}$ from the SVD).
Scale by $\sqrt{\max(1,\mathrm{rows}/\mathrm{cols})}$. **1D** params fall back to Adam.

$$
\begin{aligned}
B &\leftarrow \mu B + (1-\mu)\, G \\
O &\leftarrow \mathrm{NewtonSchulz5}(B)
          \cdot \sqrt{\max\!\bigl(1,\, m/n\bigr)} \\
W &\leftarrow W\,(1-\eta\lambda) - \eta\, O
\end{aligned}
$$

where $W\in\mathbb{R}^{m\times n}$. Ortho target:

$$
\mathrm{Ortho}(G)
= \arg\min_O
\bigl\{\|O-G\|_F :
O^{\top}O=I \;\text{or}\; OO^{\top}=I\bigr\}
$$

Helpers you should call: `newton_schulz5`, `map_leaves`.

| Pros | Cons |
|------|------|
| Strong sample efficiency on hidden matrices (NanoGPT / LLM speedruns) | Only for 2D weights — need Adam (or similar) for embeddings / vectors |
| Orthogonal updates have stable spectral norm | Extra matmuls (NS steps) → slower than AdamW per step |
| Often less LR retuning when scaling width | Newer; fewer battle-tested recipes than AdamW |
| Pair with Distill/landscape demos for geometric intuition | Easy to overfit tiny data if LR is too aggressive |


### Implement `MuonState` + `Muon`



In [ ]:
# your code here


## Smoke test

Short AdamW run to confirm your optimizer plugs into `train`.



In [ ]:
# your code here


## Curves + interactive demos

Reference LM sweep (precomputed). For *geometry*, prefer the links in the intro.

![Optimizer curves](./optimizer_curves.png)


In [ ]:
# your code here


## Takeaways

1. **SGD → Momentum** adds velocity; read [Distill](https://distill.pub/2017/momentum/).
2. **RMSProp / AdamW** add diagonal second-moment scaling (+ decay for AdamW).
3. **Muon** orthogonalizes *matrix* momentum updates; 1D params stay on Adam.
4. Same train loop for all: implement `init` + `__call__`, pass `optimizer=` into `train`.
5. Race them visually: [Gradient Optimizer Comparison](https://www.corefranciscopark.com/blog/gradient-optimizer-comparison).


---

## Exercise

1. Smoke-test `Muon()` for 50 steps (same `train(...)` call as AdamW).
2. In the [landscape demo](https://www.corefranciscopark.com/blog/gradient-optimizer-comparison), Rosenbrock + all opts — who overshoots?
3. From Distill: for fixed α, how should optimal β change as curvature λ grows?



In [ ]:
# your code here


---

**Further reading:** [Keller Jordan — Muon](https://kellerjordan.github.io/posts/muon/) has diagrams comparing convergence speeds (SGD / Adam / Muon and friends). Also read [SOAP, Muon, and Beyond: Pushing LLM Pretraining Scales](https://arxiv.org/pdf/2607.20548) from NVIDIA.
